In [2]:
from pathlib import Path
from pyspark.sql import SparkSession

PROJECT_ROOT = Path(
    r"D:\Big Data Programming Project\Final Assignment"
)

DELAY_DATA_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "delay_data"
)

spark = SparkSession.builder \
    .appName("SCNE Multi-Day Bus Delay Prediction") \
    .master("local[*]") \
    .getOrCreate()

print("Spark started successfully")
print("Spark version:", spark.version)

files = [
    str(DELAY_DATA_ROOT / "scne_delay_2025-12-26.csv"),
    str(DELAY_DATA_ROOT / "scne_delay_2025-12-27.csv"),
    str(DELAY_DATA_ROOT / "scne_delay_2025-12-28.csv")
]

delay_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(files)
)

print("Total rows:", delay_df.count())
print("Columns:", len(delay_df.columns))
print("Partitions:", delay_df.rdd.getNumPartitions())

delay_df.printSchema()

Spark started successfully
Spark version: 3.5.8
Total rows: 308885
Columns: 35
Partitions: 12
root
 |-- service_date: date (nullable = true)
 |-- source_zip: string (nullable = true)
 |-- snapshot_time: timestamp (nullable = true)
 |-- recorded_at_time: timestamp (nullable = true)
 |-- record_age_seconds: double (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- published_line_name: string (nullable = true)
 |-- direction_ref: string (nullable = true)
 |-- data_frame_ref: date (nullable = true)
 |-- dated_journey_ref: integer (nullable = true)
 |-- vehicle_journey_ref: string (nullable = true)
 |-- origin_aimed_departure_time: timestamp (nullable = true)
 |-- origin_ref: string (nullable = true)
 |-- origin_name: string (nullable = true)
 |-- destination_ref: string (nullable = true)
 |-- destination_name: string (nullable = true)
 |-- vehicle_ref: string (nullable = true)
 |-- block_ref: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: doubl

In [3]:
from pyspark.sql import functions as F

delay_df.groupBy("service_date") \
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("trip_id").alias("unique_trips"),
        F.sum(
            F.when(F.col("delay_seconds").isNull(), 1).otherwise(0)
        ).alias("missing_delay")
    ) \
    .orderBy("service_date") \
    .show()

+------------+------+------------+-------------+
|service_date|  rows|unique_trips|missing_delay|
+------------+------+------------+-------------+
|  2025-12-26| 31891|         887|            0|
|  2025-12-27|170149|        4690|            0|
|  2025-12-28|106845|        2949|            0|
+------------+------+------------+-------------+



In [4]:
# Repartition the dataset for parallel processing
delay_df = delay_df.repartition(8)

# Cache because we will reuse this dataset in later analysis
delay_df.cache()

# Trigger the cache
total_rows = delay_df.count()

print("Cached rows:", total_rows)
print("Partitions after repartition:", delay_df.rdd.getNumPartitions())

Cached rows: 308885
Partitions after repartition: 8


In [5]:
OUTPUT_PATH = str(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "spark"
    / "scne_multiday_delay"
)

delay_df.write \
    .mode("overwrite") \
    .parquet(OUTPUT_PATH)

print("Saved PySpark dataset to:")
print(OUTPUT_PATH)

Saved PySpark dataset to:
D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_multiday_delay


In [6]:
spark_df = spark.read.parquet(OUTPUT_PATH)

print("Saved rows:", spark_df.count())
print("Saved columns:", len(spark_df.columns))
print("Saved partitions:", spark_df.rdd.getNumPartitions())

spark_df.groupBy("service_date") \
    .count() \
    .orderBy("service_date") \
    .show()

Saved rows: 308885
Saved columns: 35
Saved partitions: 8
+------------+------+
|service_date| count|
+------------+------+
|  2025-12-26| 31891|
|  2025-12-27|170149|
|  2025-12-28|106845|
+------------+------+

